# Demographic Sex Ratio (Masculinity Ratio)
TThis notebook computes the **masculinity (sex) ratio** for each Spanish municipality
and year using sex-disaggregated population data from the municipal census
(*Padrón Municipal*).

The masculinity ratio is generated as a **derived demographic attribute** at the
municipality–year level and exported as a standalone dataset to ensure
methodological clarity, traceability, and reproducibility.

The resulting dataset will later be integrated with other demographic, services,
agricultural, and land-use indicators.

## Definition

According to the Spanish National Statistics Institute (INE), the masculinity
ratio is defined as the number of men per 100 women in a given population.

**Formula:**

$$
\text{Sex Ratio} =
\frac{\text{Number of men}}{\text{Number of women}}
\times 100
$$

- If $\text{Sex Ratio} > 100$, the male population exceeds the female population.  
- If $\text{Sex Ratio} < 100$, the female population exceeds the male population.

**Source:**  
INE – *Indicadores Demográficos Básicos*  
https://www.ine.es/DEFIne/concepto.htm?c=5058



In [ ]:
"""
Notebook: 03_demography_sex_ratio.ipynb
Purpose: Compute masculinity (sex) ratio per municipality and year
Input: 01_padron_clean_1996_2025.csv
Output: demography_sex_ratio_1996_2025.csv
Author: Juan Zotes
Last updated: 2026-02-05
"""


## 1. Load cleaned demographic base data

This step loads the cleaned historical municipal census dataset.
The dataset represents the **demographic base table** of the project and is not
modified in place.


In [ ]:
# Standard library
from pathlib import Path

# Third-party libraries
import pandas as pd

In [ ]:
# Base data directory (portable across Windows, Linux, Codespaces)
DATA_DIR = Path(
    r"C:\Users\juanz\OneDrive\Desktop\UCM\RURIM ESCAPE\GeoSpatial\01_Python Data Analysis\rural-migration-land-use-spain\data\demography\processed"
)

DATA_DIR


In [ ]:
# Load cleaned municipal census data
padron_file = DATA_DIR / "01_padron_clean_1996_2025.csv"

df = pd.read_csv(
    padron_file,
    dtype={"Mun_Code": str}
)

df.head()


In [ ]:
# Count municipalities and years
num_municipalities = df["Mun_Code"].nunique()
num_years = df["Year"].nunique()
print(f"Number of municipalities: {num_municipalities}")
print(f"Number of years: {num_years}")

## 2. Validate sex-disaggregated population records

Before restructuring the dataset, we verify that the cleaned census data contains
the necessary fields and categories required to compute the demographic sex ratio.


In [ ]:
# Validate required base columns
required_cols = ["Year", "Mun_Code", "Mun", "Cat", "Pop"]
missing = [col for col in required_cols if col not in df.columns]
missing


## 3. Data Restructuring for Sex Ratio Computation

The cleaned municipal census dataset is stored in **long format**, where population
counts are recorded under a categorical variable (`Cat`) with the following values:

- `Total`
- `Hombres`
- `Mujeres`

As a consequence, **male and female population counts are not stored as separate
columns**, but as separate records.

Before computing the masculinity (sex) ratio, the dataset must therefore be
**temporarily reshaped** to obtain one row per municipality–year combination with
distinct population fields for men and women.

This transformation involves:

1. Filtering the dataset to retain only records where `Cat` equals  
   **`Hombres`** or **`Mujeres`**
2. Pivoting the data from long to wide format to create two explicit fields:
   - `Population_Male`
   - `Population_Female`

This restructuring step is performed **only within this notebook** and does not
modify the original cleaned dataset, preserving its integrity and reproducibility.


In [ ]:
# Keep only male and female population records
df_sex = df[df["Cat"].isin(["Hombres", "Mujeres"])].copy()

df_sex.head()

In [ ]:
# Pivot to wide format: one row per municipality-year
sex_wide = df_sex.pivot_table(
    index=["Year", "Mun_Code", "Mun"],
    columns="Cat",
    values="Pop",
    aggfunc="sum"
).reset_index()

sex_wide.head()


## 4. Compute masculinity (sex) ratio

The masculinity ratio is calculated as the number of males per 100 females.
Municipality–year combinations with zero female population are explicitly set to
null to avoid invalid values.


In [ ]:
# Compute masculinity (sex) ratio
sex_wide["Sex_Ratio"] = (
    sex_wide["Hombres"] / sex_wide["Mujeres"]
) * 100


## 5. Quality Control

We inspect basic statistics and extreme values to ensure the indicator behaves as
expected across municipalities and years.


In [ ]:
sex_wide["Sex_Ratio"].describe()


In [ ]:
sex_wide.head()

In [ ]:
# Select final output columns
sex_ratio_df = sex_wide[
    ["Year", "Mun_Code", "Mun", "Sex_Ratio"]
].copy()



## 6. Export derived dataset

The masculinity ratio is exported as a standalone CSV file.
This modular structure allows future integration with other demographic,
agricultural, and land-use indicators without compromising traceability.


In [ ]:
DERIVED_DIR = Path(
    r"C:\Users\juanz\OneDrive\Desktop\UCM\RURIM ESCAPE\GeoSpatial\01_Python Data Analysis\rural-migration-land-use-spain\data\demography\derived"
)

# Export derived indicator
output_file = DERIVED_DIR / "demography_sex_ratio_1996_2025.csv"

sex_ratio_df.to_csv(output_file, index=False)
